In [78]:
import sqlite3
import pandas as pd

# Setting up the database. DO NOT edit the code given below
from techsmart_db_setup import setup_database
setup_database() 
conn = sqlite3.connect('techsmart.db')

# Load and display tables
tables = ['Employee_Records', 'Product_Details', 'Customer_Demographics', 'Sales_Transactions']
for table in tables:
    query = f"SELECT * FROM {table} LIMIT 5"
    df = pd.read_sql_query(query, conn)
    print(f"\n{table}:")
    display(df)

✅ Database setup complete: Tables created and populated with data!

Employee_Records:


,employee_id,role,store_location,sales_performance
0,1,Cashier,New York,NaN
1,2,Cashier,Los Angeles,3000
2,3,Supervisor,Phoenix,NaN
3,4,Manager,Chicago,3000
4,5,Manager,Phoenix,3000



Product_Details:


,product_id,product_name,category,price,stock
0,106,Tablet,Electronics,one hundred,200.0
1,101,Keyboard,Accessories,fifty,100.0
2,108,Keyboard,Accessories,fifty,500.0
3,104,Speaker,Electronics,500,150.0
4,107,Speaker,Electronics,fifty,300.0



Customer_Demographics:


,customer_id,age,gender,location,loyalty_program
0,1,35,F,San Antonio,NaN
1,2,50,M,London,No
2,3,37,M,Austin,No
3,4,NaN,M,San Antonio,NaN
4,5,28,M,San Antonio,NaN



Sales_Transactions:


,transaction_id,customer_id,product_id,employee_id,quantity,total_amount,sale_date
0,1,13.0,105,2.0,4,90.0,2025-03-03 00:00:00
1,2,37.0,105,51.0,2,90.0,2025-03-08 00:00:00
2,3,5.0,106a,NaN,three,NaN,2025-03-01 00:00:00
3,4,35.0,109b,22.0,three,40.0,2025-03-04 00:00:00
4,5,15.0,101,45.0,2,NaN,2025-03-06 00:00:00


In [79]:
#Step 2: Identify and handle missing values:
query = """
SELECT COUNT(*) as total_rows,
       SUM(CASE WHEN price IS 'fifty' OR price = 'one hundred' THEN 1 ELSE 0 END) as missing_sales
FROM Product_Details
"""
df = pd.read_sql_query(query, conn)
display(df)

,total_rows,missing_sales
0,100,33


In [80]:
#Write queries to identify missing values in other tables
query = """
SELECT COUNT(*) as total_rows,
       SUM(CASE WHEN sales_performance IS NULL OR sales_performance = 'nan' THEN 1 ELSE 0 END) as missing_sales
FROM Employee_Records
"""
df = pd.read_sql_query(query, conn)
display(df)

,total_rows,missing_sales
0,100,11


In [81]:
query = """
SELECT COUNT(*) as total_rows,
       SUM(CASE WHEN age IS NULL OR age = 'nan' THEN 1 ELSE 0 END)  as missing_age,
       SUM(CASE WHEN loyalty_program IS NULL OR loyalty_program = 'nan' THEN 1 ELSE 0 END ) as missing_loyalty_program
FROM Customer_Demographics
"""
df = pd.read_sql_query(query, conn)
display(df)

,total_rows,missing_age,missing_loyalty_program
0,100,12,35


In [82]:
query = """
SELECT COUNT(*) as total_rows,
       SUM(CASE WHEN product_id GLOB '*[a-zA-Z]*' THEN 1 ELSE 0 END)as missing_product_id,
       SUM (CASE WHEN employee_id IS NULL OR employee_id ='NaN'  THEN 1 ELSE 0 END)as missing_employee_id,
       SUM ( CASE WHEN quantity IS NULL OR quantity ='three'  THEN 1 ELSE 0 END ) as missing_quantity
FROM Sales_Transactions
"""
df = pd.read_sql_query(query, conn)
display(df)

,total_rows,missing_product_id,missing_employee_id,missing_quantity
0,100,25,7,32


In [83]:
# Clean inconsistent data formats:
# Example: Standardize sales_performance in Employee_Records
query = """
UPDATE Employee_Records
SET sales_performance = CASE
    WHEN sales_performance = 'nan' THEN NULL
    WHEN sales_performance = 'five thousand' THEN '5000'
    ELSE sales_performance
END
"""
cursor = conn.cursor()
cursor.execute(query)
conn.commit()

In [84]:
# Clean inconsistent data in Product_Details
# Your turn: Clean inconsistent data Product_Details
query = """
UPDATE Product_Details
SET price = CASE
    WHEN price = 'one hundred' THEN '100'
    WHEN price = 'fifty' THEN '50'
    WHEN price = 'nan' THEN NULL
    ELSE price
END
"""
cursor = conn.cursor()
cursor.execute(query)
conn.commit()

In [85]:
query = """
SELECT DISTINCT price FROM Product_Details
"""
df = pd.read_sql_query(query, conn)
display(df)

,price
0,100
1,50
2,500
3,NaN
4,25
5,40
6,200
7,300
8,150


In [86]:
# Your turn: Clean inconsistent data Customer_Demographics
query = """
UPDATE Customer_Demographics
SET loyalty_program = CASE
    WHEN UPPER(loyalty_program) = 'NAN' THEN NULL
    ELSE loyalty_program    
END,
 age = CASE
    WHEN age GLOB '*[a-zA-Z]*' THEN NULL
    ELSE age
END
"""
cursor = conn.cursor()
cursor.execute(query)
conn.commit()

In [87]:
query = """
UPDATE Customer_Demographics
SET age = (SELECT ROUND(AVG(CAST(age AS FLOAT))) FROM Customer_Demographics WHERE age IS NOT NULL)
WHERE age IS NULL
"""
cursor = conn.cursor()
cursor.execute(query)
conn.commit()

In [88]:
#هنا بنشوف ايش بدنا ننضف في الجدول
query = """
SELECT DISTINCT loyalty_program FROM Customer_Demographics"""
df = pd.read_sql_query(query, conn)
display(df)

,loyalty_program
0,NaN
1,No
2,Yes


In [89]:
query = "SELECT DISTINCT age FROM Customer_Demographics"
df = pd.read_sql_query(query, conn)
display(df)

,age
0,35
1,50
2,37
3,37.0
4,28
5,25
6,43
7,42
8,41
9,40


In [90]:
# Your turn: Clean inconsistent data Sales_Transactions
query = """
UPDATE Sales_Transactions
SET product_id = CASE
    WHEN product_id = '106a' THEN '106'
    WHEN product_id = '109b' THEN '109'
    ELSE product_id
END,
 quantity = CASE
 WHEN quantity = 'three' THEN '3'
 ELSE quantity
 END"""


cursor = conn.cursor()
cursor.execute(query)
conn.commit()


In [91]:
df = pd.read_sql_query("""
SELECT
    SUM(quantity IS NULL) AS real_nulls,
    SUM(quantity = 'NaN') AS text_nan
FROM Sales_Transactions
""", conn)
display(df)

,real_nulls,text_nan
0,17,0


In [92]:

query = "SELECT DISTINCT quantity FROM Sales_Transactions"
df = pd.read_sql_query(query, conn)
display(df)


,quantity
0,4
1,2
2,3
3,1
4,NaN


In [93]:
query = "SELECT DISTINCT product_id FROM Sales_Transactions"
df = pd.read_sql_query(query, conn)
display(df)

,product_id
0,105
1,106
2,109
3,101
4,103
5,102
6,104


In [94]:
df = pd.read_sql_query("SELECT COUNT(*) AS total_rows FROM Sales_Transactions", conn)
display(df)

,total_rows
0,100


In [95]:
cursor = conn.cursor()
cursor.execute("DELETE FROM Sales_Transactions WHERE quantity IS NULL")
print("الصفوف المحذوفة:", cursor.rowcount)#✅ ليش الحذف منطقي
# الكمية جزء أساسي من الصفقة، وإذا عوّضناها برقم من عندنا منكون اخترعنا معلومة.
# الصف بدون كمية ما بنقدر نحسب منه مبيعات صح.
# وهاد بيوافق خطتك بتنويع الأساليب: NULL بالسعر، تعويض بالـ age، وحذف هون.
conn.commit()

الصفوف المحذوفة: 17


In [96]:
df = pd.read_sql_query("SELECT COUNT(*) AS remaining_nulls FROM Sales_Transactions WHERE quantity IS NULL", conn)
display(df)

,remaining_nulls
0,0


In [97]:
cursor = conn.cursor()
cursor.execute("DELETE FROM Sales_Transactions WHERE employee_id IS NULL")
print("الصفوف المحذوفة:", cursor.rowcount)#✅ 
conn.commit()

الصفوف المحذوفة: 5


In [98]:
df = pd.read_sql_query("SELECT COUNT(*) AS remaining_nulls FROM Sales_Transactions WHERE employee_id IS NULL", conn)
display(df)

,remaining_nulls
0,0


In [99]:
df = pd.read_sql_query("""
SELECT
    COUNT(*) AS total_rows,
    SUM(quantity IS NULL) AS quantity_nulls,
    SUM(employee_id IS NULL) AS employee_nulls
FROM Sales_Transactions
""", conn)
display(df)

,total_rows,quantity_nulls,employee_nulls
0,78,0,0


In [100]:
df = pd.read_sql_query("""
SELECT DISTINCT product_id
FROM Sales_Transactions
WHERE product_id GLOB '*[a-zA-Z]*'
""", conn)
display(df)

,product_id


In [101]:
df = pd.read_sql_query("""
SELECT
    COUNT(*) AS total_rows,
    SUM(product_id IS NULL) AS product_nulls
FROM Sales_Transactions
""", conn)
display(df)

,total_rows,product_nulls
0,78,0


In [102]:
df = pd.read_sql_query("""
SELECT
    SUM(total_amount IS NULL) AS real_nulls,
    SUM(total_amount = 'NaN') AS text_nan,
    SUM(total_amount GLOB '*[a-zA-Z]*') AS has_letters
FROM Sales_Transactions
""", conn)
display(df)

,real_nulls,text_nan,has_letters
0,11,0,0


In [103]:
df = pd.read_sql_query("""
SELECT s.transaction_id, s.product_id, s.quantity, p.price
FROM Sales_Transactions s
JOIN Product_Details p ON s.product_id = p.product_id
WHERE s.total_amount IS NULL
""", conn)
display(df)

,transaction_id,product_id,quantity,price
0,5,101,2,NaN
1,5,101,2,100
2,5,101,2,150
3,5,101,2,200
4,5,101,2,25
...,...,...,...,...
88,99,106,4,100
89,99,106,4,300
90,99,106,4,50
91,99,106,4,500


In [104]:
df = pd.read_sql_query("""
SELECT product_id, COUNT(*) AS how_many
FROM Product_Details
GROUP BY product_id
HAVING COUNT(*) > 1
""", conn)
display(df)

,product_id,how_many
0,101,7
1,102,10
2,103,10
3,104,9
4,105,8
5,106,6
6,107,15
7,108,5
8,109,15
9,110,15


In [105]:
df = pd.read_sql_query("""
SELECT * FROM Product_Details
WHERE product_id = '101'
""", conn)
display(df)

,product_id,product_name,category,price,stock
0,101,Keyboard,Accessories,50,100.0
1,101,Camera,Electronics,150,50.0
2,101,Tablet,Electronics,NaN,150.0
3,101,Mouse,Electronics,50,NaN
4,101,Mouse,Accessories,100,100.0
5,101,Charger,Accessories,200,200.0
6,101,Camera,Electronics,25,300.0


In [106]:
df = pd.read_sql_query("""
SELECT AVG(total_amount) AS avg_amount
FROM Sales_Transactions
WHERE total_amount IS NOT NULL
""", conn)
display(df)

,avg_amount
0,52.089552


In [107]:
cursor = conn.cursor()
cursor.execute("DELETE FROM Sales_Transactions WHERE total_amount IS NULL")
print("الصفوف المحذوفة:", cursor.rowcount)
conn.commit()

الصفوف المحذوفة: 11


In [108]:
df = pd.read_sql_query("SELECT COUNT(*) AS remaining_nulls FROM Sales_Transactions WHERE total_amount IS NULL", conn)
display(df)

,remaining_nulls
0,0


In [109]:
df = pd.read_sql_query("""
SELECT loyalty_program, COUNT(*) AS how_many
FROM Customer_Demographics
GROUP BY loyalty_program
""", conn)
display(df)

,loyalty_program,how_many
0,NaN,35
1,No,32
2,Yes,33


In [110]:
df = pd.read_sql_query("""
SELECT
    SUM(loyalty_program IS NULL) AS real_nulls,
    SUM(UPPER(loyalty_program) = 'NAN') AS text_nan
FROM Customer_Demographics
""", conn)
display(df)

,real_nulls,text_nan
0,35,0


In [111]:
#  Analyze employee performance by location:
query = """
WITH emp_sales AS (
    SELECT store_location, 
           AVG(CAST(sales_performance AS FLOAT)) as avg_sales,
           COUNT(*) as employee_count
    FROM Employee_Records
    WHERE sales_performance IS NOT NULL
    GROUP BY store_location
)
SELECT store_location, avg_sales, employee_count,
       RANK() OVER (ORDER BY avg_sales DESC) as location_rank
FROM emp_sales
ORDER BY avg_sales DESC
"""
df = pd.read_sql_query(query, conn)
display(df)

,store_location,avg_sales,employee_count,location_rank
0,Chicago,3875.000000,8,1
1,Phoenix,3312.500000,16,2
2,London,3214.285714,14,3
3,New York,2823.529412,17,4
4,Los Angeles,2692.307692,13,5
5,Paris,2600.000000,10,6
6,Houston,2363.636364,11,7


In [112]:
# Identify top-selling products by category:
query = """
SELECT *
FROM (
    SELECT p.category,
           p.product_name,
           SUM(s.quantity) AS total_sold,
           RANK() OVER (
               PARTITION BY p.category
               ORDER BY SUM(s.quantity) DESC
           ) AS rank_in_category
    FROM Sales_Transactions s
    JOIN Product_Details p ON s.product_id = p.product_id
    GROUP BY p.category, p.product_name
)
WHERE rank_in_category <= 3
ORDER BY category, total_sold DESC
"""
df = pd.read_sql_query(query, conn)
display(df)

,category,product_name,total_sold,rank_in_category
0,Accessories,Keyboard,185,1
1,Accessories,Charger,165,2
2,Accessories,Speaker,90,3
3,Electronics,Tablet,226,1
4,Electronics,Mouse,149,2
5,Electronics,Headphones,107,3


In [113]:
# Step 3: Try it yourself Analyze customer purchasing behavior تحليل سلوك الزبائن:


# # Your turn: Write a query to analyze customer purchasing behavior
# # Hint: Join Customer_Demographics with Sales_Transactions and use window functions
query = """
SELECT *
FROM (
    SELECT c.customer_id,
           c.loyalty_program,
           COUNT(*) AS purchase_count,
           SUM(s.total_amount) AS total_spent,
           RANK() OVER (ORDER BY SUM(s.total_amount) DESC) AS rank_in_category
    FROM Customer_Demographics c
    
    JOIN Sales_Transactions s ON c.customer_id = s.customer_id
    GROUP BY c.customer_id,c.loyalty_program
    )
    
    ORDER BY customer_id, total_spent DESC
"""
df = pd.read_sql_query(query, conn)
display(df)

,customer_id,loyalty_program,purchase_count,total_spent,rank_in_category
0,1,NaN,2,120.0,12
1,2,No,4,210.0,6
2,3,No,3,170.0,8
3,4,NaN,5,250.0,4
4,5,NaN,2,70.0,16
5,10,NaN,1,50.0,17
6,12,Yes,5,370.0,2
7,13,NaN,8,440.0,1
8,15,Yes,3,130.0,11
9,35,Yes,2,80.0,14


In [114]:
#يعني لو زبون عنده 5 صفوف (5 معاملات) قبل التجميع، بعد التجميع COUNT(*) رح تساوي 5 — يعني عدد مرات الشراء تبع هاد الزبون.
query = """
WITH customer_behavior AS (
    SELECT c.customer_id,
           c.loyalty_program,
           COUNT(*) AS purchase_count,
           SUM(s.total_amount) AS total_spent
    FROM Customer_Demographics c
    JOIN Sales_Transactions s ON c.customer_id = s.customer_id
    GROUP BY c.customer_id, c.loyalty_program
)
SELECT loyalty_program,
       COUNT(*) AS num_customers,
       AVG(purchase_count) AS avg_purchases,
       AVG(total_spent) AS avg_spent
FROM customer_behavior
GROUP BY loyalty_program
"""
df = pd.read_sql_query(query, conn)
display(df)
# 🔹 avg_purchases (متوسط عدد مرات الشراء)

# بيجاوب على سؤال: "بالمتوسط، كم مرة يشتري الزبون؟"

# هو متوسط عمود purchase_count (يلي حسبناه قبل = COUNT(*) لكل زبون). يعني لو فئة "Yes" عندها avg_purchases = 4، معناها بالمتوسط كل زبون مشترك ببرنامج الولاء بيشتري 4 مرات.

# ➡️ هاد مقياس التكرار / النشاط (هل الزبون بيرجع يشتري كذا مرة؟)

# 🔹 avg_spent (متوسط المبلغ المصروف)

# بيجاوب على سؤال: "بالمتوسط، كم فلوس يصرف الزبون إجمالاً؟"

# هو متوسط عمود total_spent (يلي حسبناه قبل = SUM(total_amount) لكل زبون). يعني لو فئة "Yes" عندها avg_spent = 300، معناها بالمتوسط كل زبون مشترك صرف 300 بالمجموع (عبر كل معاملاته).

# ➡️ هاد مقياس القيمة المالية (كم فلوس جابوا للمتجر؟)

,loyalty_program,num_customers,avg_purchases,avg_spent
0,NaN,9,3.666667,187.777778
1,No,4,4.000000,200.000000
2,Yes,4,3.000000,182.500000


In [115]:
# 1: Identify slow-running queries:


# Example: Time a complex query
import time
start_time = time.time()
query = """
WITH StoreSales AS (
   SELECT
       e.store_location,
       e.employee_id,
       e.role,
       p.category,
       c.loyalty_program,
       COUNT(s.transaction_id) AS total_sales,
       SUM(s.quantity) AS total_units_sold,
       SUM(s.total_amount) AS total_revenue
   FROM Sales_Transactions s
   JOIN Employee_Records e ON s.employee_id = e.employee_id
   JOIN Product_Details p ON s.product_id = p.product_id
   JOIN Customer_Demographics c ON s.customer_id = c.customer_id
   WHERE p.category IN ('Electronics', 'Accessories')
   GROUP BY e.store_location, e.employee_id, e.role, p.category, c.loyalty_program
),

StoreRankings AS (
   SELECT
       store_location,
       category,
       SUM(total_revenue) AS store_revenue
   FROM StoreSales
   GROUP BY store_location, category
)

SELECT
   ss.store_location,
   ss.employee_id,
   ss.role,
   ss.category,
   ss.loyalty_program,
   ss.total_sales,
   ss.total_units_sold,
   ss.total_revenue
FROM StoreSales ss
JOIN StoreRankings sr ON ss.store_location = sr.store_location AND ss.category = sr.category
ORDER BY ss.store_location, ss.total_revenue DESC;

"""
df = pd.read_sql_query(query, conn)
end_time = time.time()
print(f"Query execution time: {end_time - start_time} seconds")

Query execution time: 0.006153106689453125 seconds


In [116]:
cursor.execute("CREATE INDEX IF NOT EXISTS idx_product_category ON Product_Details(category)")
conn.commit()

In [117]:
# كل JOIN بيربط عمودين. عشان نسرّع المطابقة، منحط index على كل عمود مستخدم بشرط JOIN، بكلا الجدولين (المصدر والهدف).
cursor.execute("CREATE INDEX IF NOT EXISTS idx_sales_employee ON Sales_Transactions(employee_id)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_employee_id ON Employee_Records(employee_id)")

cursor.execute("CREATE INDEX IF NOT EXISTS idx_sales_product ON Sales_Transactions(product_id)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_product_id ON Product_Details(product_id)")

cursor.execute("CREATE INDEX IF NOT EXISTS idx_sales_customer ON Sales_Transactions(customer_id)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_customer_id ON Customer_Demographics(customer_id)")

conn.commit()

In [118]:
query="""
 WITH StoreSales AS (
    SELECT
        e.store_location,
        e.employee_id,
        e.role,
        p.category,
        c.loyalty_program,
        COUNT(s.transaction_id) AS total_sales,
        SUM(s.quantity) AS total_units_sold,
        SUM(s.total_amount) AS total_revenue,
        SUM(SUM(s.total_amount)) OVER (PARTITION BY e.store_location, p.category) AS store_category_revenue
    FROM Sales_Transactions s
    JOIN Employee_Records e ON s.employee_id = e.employee_id
    JOIN Product_Details p ON s.product_id = p.product_id
    JOIN Customer_Demographics c ON s.customer_id = c.customer_id
    WHERE p.category IN ('Electronics', 'Accessories')
    GROUP BY e.store_location, e.employee_id, e.role, p.category, c.loyalty_program
)
SELECT *
FROM StoreSales
ORDER BY store_location, total_revenue DESC
"""
df = pd.read_sql_query(query, conn)
display(df)

,store_location,employee_id,role,category,loyalty_program,total_sales,total_units_sold,total_revenue,store_category_revenue
0,Chicago,17,Sales Associate,Electronics,NaN,19,47,750.0,1250.0
1,Chicago,17,Sales Associate,Accessories,NaN,15,35,600.0,960.0
2,Chicago,17,Sales Associate,Electronics,No,11,23,500.0,1250.0
3,Chicago,17,Sales Associate,Accessories,No,8,16,360.0,960.0
4,Houston,45,Manager,Electronics,No,14,28,1060.0,2380.0
5,Houston,45,Manager,Accessories,No,12,24,880.0,1960.0
6,Houston,45,Manager,Electronics,NaN,12,28,870.0,2380.0
7,Houston,45,Manager,Accessories,NaN,10,23,720.0,1960.0
8,Houston,45,Manager,Electronics,Yes,5,15,450.0,2380.0
9,Houston,45,Manager,Accessories,Yes,4,12,360.0,1960.0


In [119]:
start_time = time.time()
query = """
WITH StoreSales AS (
    SELECT
        e.store_location,
        e.employee_id,
        e.role,
        p.category,
        c.loyalty_program,
        COUNT(s.transaction_id) AS total_sales,
        SUM(s.quantity) AS total_units_sold,
        SUM(s.total_amount) AS total_revenue,
        SUM(SUM(s.total_amount)) OVER (PARTITION BY e.store_location, p.category) AS store_category_revenue
    FROM Sales_Transactions s
    JOIN Employee_Records e ON s.employee_id = e.employee_id
    JOIN Product_Details p ON s.product_id = p.product_id
    JOIN Customer_Demographics c ON s.customer_id = c.customer_id
    WHERE p.category IN ('Electronics', 'Accessories')
    GROUP BY e.store_location, e.employee_id, e.role, p.category, c.loyalty_program
)
SELECT *
FROM StoreSales
ORDER BY store_location, total_revenue DESC
"""

df = pd.read_sql_query(query, conn)
end_time = time.time()
print(f"Query execution time: {end_time - start_time} seconds")
display(df)

Query execution time: 0.0035376548767089844 seconds


,store_location,employee_id,role,category,loyalty_program,total_sales,total_units_sold,total_revenue,store_category_revenue
0,Chicago,17,Sales Associate,Electronics,NaN,19,47,750.0,1250.0
1,Chicago,17,Sales Associate,Accessories,NaN,15,35,600.0,960.0
2,Chicago,17,Sales Associate,Electronics,No,11,23,500.0,1250.0
3,Chicago,17,Sales Associate,Accessories,No,8,16,360.0,960.0
4,Houston,45,Manager,Electronics,No,14,28,1060.0,2380.0
5,Houston,45,Manager,Accessories,No,12,24,880.0,1960.0
6,Houston,45,Manager,Electronics,NaN,12,28,870.0,2380.0
7,Houston,45,Manager,Accessories,NaN,10,23,720.0,1960.0
8,Houston,45,Manager,Electronics,Yes,5,15,450.0,2380.0
9,Houston,45,Manager,Accessories,Yes,4,12,360.0,1960.0


In [120]:
# تم إجراء تحليل شامل لبيانات TechMart لفهم سلوك الزبائن الشرائي وعلاقته ببرنامج الولاء. أحد أهم الاستعلامات المستخدمة قام بربط جدولي Customer_Demographics وSales_Transactions، وحساب متوسط عدد مرات الشراء ومتوسط إجمالي الإنفاق لكل فئة من فئات الاشتراك في برنامج الولاء (مشترك، غير مشترك، وغير معروف).

# كشف الاستعلام عن نتيجة غير متوقعة: الزبائن غير المشتركين في برنامج الولاء (No) سجلوا متوسط عدد مشتريات أعلى قليلاً (4.0 مرة) ومتوسط إنفاق أعلى (200 دولار) مقارنة بالزبائن المشتركين (Yes)، الذين سجلوا متوسط 3.0 مرات شراء و182.5 دولار إنفاق. هذه النتيجة تتعارض مع الافتراض الشائع بأن برامج الولاء تزيد من نشاط الزبون الشرائي. ومع ذلك، يجب الإشارة إلى أن حجم العينة صغير جداً (4 زبائن فقط في كل فئة)، إضافة إلى أن 9 زبائن (الأغلبية) لم تُحدَّد حالتهم في برنامج الولاء بسبب بيانات ناقصة، مما يحد من قوة الاستنتاج النهائي.

# **التوصية العملية:** يُنصح بإجراء مراجعة شاملة لفعالية برنامج الولاء الحالي، بدءاً بتحسين جمع البيانات لضمان تسجيل حالة الاشتراك لجميع الزبائن، ثم إعادة هذا التحليل على عينة أكبر وأكثر اكتمالاً لتحديد ما إذا كان البرنامج يستحق الاستثمار المستمر فيه أو يحتاج لإعادة تصميم لتحفيز الزبائن فعلياً على الشراء بشكل متكرر.

In [121]:
# Analyze employee performance by location:
query="""
WITH emp_sales AS (
    SELECT store_location, 
           AVG(CAST(sales_performance AS FLOAT)) as avg_sales,
           COUNT(*) as employee_count
    FROM Employee_Records
    WHERE sales_performance IS NOT NULL
    GROUP BY store_location
)
SELECT store_location, avg_sales, employee_count,
       RANK() OVER (ORDER BY avg_sales DESC) as location_rank
FROM emp_sales
"""
df = pd.read_sql_query(query, conn)
display(df)


,store_location,avg_sales,employee_count,location_rank
0,Chicago,3875.000000,8,1
1,Phoenix,3312.500000,16,2
2,London,3214.285714,14,3
3,New York,2823.529412,17,4
4,Los Angeles,2692.307692,13,5
5,Paris,2600.000000,10,6
6,Houston,2363.636364,11,7


In [122]:
# Identify top-selling products by category: .
# هل بنفع لمن اعمل سيتياي اختار من جدولين عادي و لكن تحت عامل ليش سيلكت لكل الجداول وفتح 

In [123]:
# Identify top-selling products by category:
query = """
SELECT *
FROM (
    SELECT p.category,
           p.product_name,
           SUM(s.quantity) AS total_sold,
           RANK() OVER (
               PARTITION BY p.category
               ORDER BY SUM(s.quantity) DESC
           ) AS rank_in_category
    FROM Sales_Transactions s
    JOIN Product_Details p ON s.product_id = p.product_id
    GROUP BY p.category, p.product_name
)
WHERE rank_in_category <= 3
ORDER BY category, total_sold DESC
"""
df = pd.read_sql_query(query, conn)
display(df)

,category,product_name,total_sold,rank_in_category
0,Accessories,Keyboard,185,1
1,Accessories,Charger,165,2
2,Accessories,Speaker,90,3
3,Electronics,Tablet,226,1
4,Electronics,Mouse,149,2
5,Electronics,Headphones,107,3


In [124]:
query="""
WITH ranked_products AS (
    SELECT p.category,
           p.product_name,
           SUM(s.quantity) AS total_sold,
           RANK() OVER (
               PARTITION BY p.category
               ORDER BY SUM(s.quantity) DESC
           ) AS rank_in_category
    FROM Sales_Transactions s
    JOIN Product_Details p ON s.product_id = p.product_id
    GROUP BY p.category, p.product_name
)
SELECT *
FROM ranked_products
WHERE rank_in_category <= 3
ORDER BY category, total_sold DESC
"""
df = pd.read_sql_query(query, conn)
display(df)

,category,product_name,total_sold,rank_in_category
0,Accessories,Keyboard,185,1
1,Accessories,Charger,165,2
2,Accessories,Speaker,90,3
3,Electronics,Tablet,226,1
4,Electronics,Mouse,149,2
5,Electronics,Headphones,107,3
